# From notebook to script: many files
Python practice from [The Joinery](https://thejoinery.dev/). Run each cell with **Shift + Enter**, from the top. The explanations are on the guide page: https://thejoinery.dev/#/projects/python-files

## 1. A function to read one day

In [ ]:
import pandas as pd

BASE = "https://thejoinery.dev/projects/python-data/daily/"

def load_day(day):
    """Read one day's export and note which file each row came from."""
    df = pd.read_csv(f"{BASE}orders_{day}.csv")
    df["source_file"] = f"orders_{day}.csv"
    return df

monday = load_day("2025-11-24")
print(monday.shape)
print(monday.head(3))

## 2. Loop over the week

In [ ]:
days = pd.date_range("2025-11-24", "2025-11-30").strftime("%Y-%m-%d")
frames = []
for day in days:
    df = load_day(day)
    print(day, df.shape)
    frames.append(df)

## 3. Spot new columns

In [ ]:
first_columns = set(frames[0].columns)
for df in frames:
    new_columns = set(df.columns) - first_columns
    if new_columns:
        print(df["source_file"].iloc[0], "has new columns:", new_columns)

## 4. Combine the files

In [ ]:
week = pd.concat(frames, ignore_index=True)
print(week.shape)
print(sum(len(df) for df in frames))
print(week["delivery_method"].isna().sum(), "rows have no delivery_method")

## 5. Check the result

In [ ]:
print(week["order_id"].is_unique)
by_file = (week
           .groupby("source_file")
           .agg(orders=("order_id", "count"), revenue=("order_total", "sum"))
           .round(2))
print(by_file)

## 6. Turn it into a script

In [ ]:
%%writefile combine_week.py
"""Combine a week of daily order exports into one file."""
import pandas as pd

BASE = "https://thejoinery.dev/projects/python-data/daily/"
DAYS = pd.date_range("2025-11-24", "2025-11-30").strftime("%Y-%m-%d")


def load_day(day):
    df = pd.read_csv(f"{BASE}orders_{day}.csv")
    df["source_file"] = f"orders_{day}.csv"
    return df


def main():
    week = pd.concat([load_day(day) for day in DAYS], ignore_index=True)
    week.to_csv("week.csv", index=False)
    print(f"Combined {len(DAYS)} files into week.csv: {len(week)} rows")


if __name__ == "__main__":
    main()

In [ ]:
!python combine_week.py